# NLP Exercises (Part 2)

We have 2 exercises in this section. The exercises are:

4. Build your own Bag Of Words implementation using tokenizer created before.
5. Build a 5-gram model and clean up the results.

## Exercise 4. Build your own Bag Of Words implementation using tokenizer created before 

You need to implement following methods:

- ``fit_transform`` - gets a list of strings and returns matrix with it's BoW representation
- ``get_features_names`` - returns list of words corresponding to columns in BoW

In [15]:
import numpy as np
import spacy

class BagOfWords:
    """Basic BoW implementation."""
    
    __nlp = spacy.load("en_core_web_sm")
    __bow_list = []
    
    def __init__(self):
        self.__bow_list = []

    def __tokenize(self, text: str):
        return [
            token.text.lower()
            for token in self.__nlp(text)
            if token.is_alpha
        ] 
    
    def fit_transform(self, corpus: list):
        """Transform list of strings into BoW array.

        Parameters
        ----------
        corpus: List[str]
                Corpus of texts to be transforrmed

        Returns
        -------
        np.array
                Matrix representation of BoW

        """
        tokenized_corpus = [self.__tokenize(text) for text in corpus]

        self.__bow_list = sorted({
            word
            for document in tokenized_corpus
            for word in document
        })

        word_to_index = {
            word: index
            for index, word in enumerate(self.__bow_list)
        }

        bow = np.zeros((len(corpus), len(self.__bow_list)), dtype=int)

        for row_index, document in enumerate(tokenized_corpus):
            for word in document:
                bow[row_index][word_to_index[word]] += 1

        return bow     
        return None
      

    def get_feature_names(self) -> list:
        """Return words corresponding to columns of matrix.

        Returns
        -------
        List[str]
                Words being transformed by fit function

        """   
        return self.__bow_list

corpus = [
     'Bag Of Words is based on counting',
     'words occurences throughout multiple documents.',
     'This is the third document.',
     'As you can see most of the words occur only once.',
     'This gives us a pretty sparse matrix, see below. Really, see below',
]    
    
vectorizer = BagOfWords()

X = vectorizer.fit_transform(corpus)
print(X)

vectorizer.get_feature_names()
len(vectorizer.get_feature_names())

print(vectorizer.get_feature_names())

[[0 0 1 1 0 0 1 0 0 0 1 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0]
 [0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0]
 [0 1 0 0 0 1 0 0 0 0 0 0 1 0 1 0 1 0 1 1 0 0 1 0 1 0 0 0 0 1 1]
 [1 0 0 0 2 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 1 1 2 1 0 0 1 0 1 0 0]]
['a', 'as', 'bag', 'based', 'below', 'can', 'counting', 'document', 'documents', 'gives', 'is', 'matrix', 'most', 'multiple', 'occur', 'occurences', 'of', 'on', 'once', 'only', 'pretty', 'really', 'see', 'sparse', 'the', 'third', 'this', 'throughout', 'us', 'words', 'you']


## Exercise 5. Build a 5-gram model and clean up the results.

There are three tasks to do:
1. Use 5-gram model instead of 3.
2. Change to capital letter each first letter of a sentence.
3. Remove the whitespace between the last word in a sentence and . ! or ?.

Hint: for 2. and 3. implement a function called ``clean_generated()`` that takes the generated text and fix both issues at once. It could be easier to fix the text after it's generated rather then doing some changes in the while loop.

In [16]:
import re
import random

with open("datasets/trump.txt", "r", encoding="utf-8") as file:
    text = file.read()

wall_street = re.findall(r"\b\w+\b|[.!?]", text)


tokens = wall_street

def cleanup():
    compiled_pattern = re.compile("^[a-zA-Z0-9.!?]")
    clean = list(filter(compiled_pattern.match,tokens))
    return clean
tokens = [token.lower() for token in cleanup()]

def build_ngrams():
    ngrams = []
    for i in range(len(tokens)-N+1):
        ngrams.append(tokens[i:i+N])
    return ngrams

def ngram_freqs(ngrams):
    counts = {}

    for ngram in ngrams:
        token_seq  = SEP.join(ngram[:-1])
        last_token = ngram[-1]

        if token_seq not in counts:
            counts[token_seq] = {}

        if last_token not in counts[token_seq]:
            counts[token_seq][last_token] = 0

        counts[token_seq][last_token] += 1;

    return counts

def next_word(text, N, counts):
    token_seq = SEP.join(text.split()[-(N-1):])

    if token_seq not in counts:
        token_seq = random.choice(list(counts.keys()))

    choices = counts[token_seq].items()

    total = sum(weight for choice, weight in choices)
    r = random.uniform(0, total)
    upto = 0

    for choice, weight in choices:
        upto += weight
        if upto > r:
            return choice

    assert False

In [17]:
def clean_generated(text):
    text = re.sub(r"\s+([.!?])", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(
        r"(^|[.!?]\s+)([a-z])",
        lambda match: match.group(1) + match.group(2).upper(),
        text
    )
    return text

N = 5
SEP = " "
sentence_count = 5

ngrams = build_ngrams()
counts = ngram_freqs(ngrams)

start_seq = random.choice(list(counts.keys()))
generated = start_seq

sentences = 0
while sentences < sentence_count:
    generated += SEP + next_word(generated, N, counts)
    sentences += 1 if generated.endswith(('.', '!', '?')) else 0

generated = clean_generated(generated)

print(generated)

Part of a highly successful raid that generated large amounts of vital intelligence that will lead to many more victories in the future against our enemy. Ryan s legacy is etched into eternity. Thank you. And a lifetime ban on becoming lobbyists for a foreign government. We have undertaken a historic effort to massively reduce job crushing regulations creating a deregulation task force inside of every government agency.
